In [1]:
pip install wikipedia transformers torch sentencepiece -q

  Preparing metadata (setup.py) ... done


In [26]:
import wikipedia

#method to look up on wikipedia
def search_wikipedia(topic: str) ->str :
    """
    This is our first tool. It searches for a topic on wikipedia and returns
    the content of the page.
    """
    print(f"Searching Wikipedia for: {topic}")

    try:
      # we use wikipedia.page() to get the full page object
      # auto_suggest=FAlse prevents it from guessin a different topic
      page = wikipedia.page(topic,auto_suggest=False, redirect=True)
      return page.content
    except wikipedia.exceptions.PageError:
      # this happens when a topic is too broad (e.g., "Apple")
      return f"Error: Sorry I couldn't find a page for '{topic}'"

    except wikipedia.exceptions.DisambiguationError as e:
      # this happens when a topic is too broad (e.g., "Apple")
      return f"Error: '{topic}' is ambiguos. di you mean on of these: {e.options[:5]}"
    except Exception as e:
      return f"Unexpected Error: {e}"

print("Function search_wikipedia defined")

Function search_wikipedia defined


In [38]:
#let's test our new tool
test_content = search_wikipedia("Reinforcement learning")
print(f"Tools test output (first 200 chars): {test_content[:200]}...")

Searching Wikipedia for: Reinforcement learning
Tools test output (first 200 chars): Unexpected Error: Expecting value: line 1 column 1 (char 0)...


In [28]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [29]:
# Load the model and tokenizer (this happens only once) ---
print("Loading T5 model and tokenizer...")
model_name = "t5-small"
try:
  model = T5ForConditionalGeneration.from_pretrained(model_name)
  tokenizer = T5Tokenizer.from_pretrained(model_name)
  print("Model and tokenizer load succesfully")
except Exception as e:
  print(f"Unexpected error loading model: {e}")
  model = None


#Now we'll define the summarize_text function
#---define the summarize_text function

def summarize_text(text: str) -> str:
  """
  This is our second tool. It uses a T5 model to summarize text
  """
  if model is None:
    return "Error: Summarization model is not available"

  print(f"Summarizing text with T5 model: {model_name}")
  #T5 models require a prefix for different tasks. For summarization, it's "summarize: "
  text_to_summarize = "summarize: " + text

  #1. tokenize: Convert the text into a format the model understands
  inputs = tokenizer.encode(text_to_summarize, return_tensors="pt", max_length=512, truncation=True)
  #2. Generate: Ask the model to generate the summary
  summary_ids =  model.generate(
      inputs,
      max_length=150, #The longest the summary can be
      min_length=40, # the shortest it can be
      length_penalty=2.0, #encourages the model not to use too many words
      num_beams=4, # A technique for generating higher quality text
      early_stopping=True
      )
  #3. Decode: Convert the model's output back into text
  summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
  return summary


Loading T5 model and tokenizer...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model and tokenizer load succesfully


In [33]:
class WebResearchAgent:
  def __init__(self):
    self.memory = {} # A simple dictionary to store results steps

  def run(self, research_topic: str) -> str:
    """
    The main execution loop of the agent
    """
    print(f"\n Agent starting job for topic: {research_topic}")

    #---- STEP 1: SEARCH ---
    print(f"\n[Thinking] I need to find information on the web for this topic. {research_topic} I'll use the search_wikipedia tool.")
    # Action
    article_content =  search_wikipedia(research_topic)
    print(article_content)
    self.memory["raw_article"] = article_content

    #Observation
    if "Error:" in article_content:
      print(f"[Observation] The search tool failed. Aborting job.")
      return article_content #return the error message

    print(f"[Observation] The search tool succesfully retrieved the article. It's {len(article_content)}")

    #---- STEP 2: SUMMARIZE ---
    print("\n[Thinking] The article is too long to read. I need to summarize it. I'll use the summarization_text tool")

    # Action
    summary = summarize_text(article_content)
    self.memory["summary"] = summary

    #Observation
    print(f"[Observation] The summarization tool succesfully summarized the article. It's {len(summary)}")
    print("Agent job finished")
    return summary

  print ("WebResearchAgent class defined")


WebResearchAgent class defined


In [34]:
# main execution block
agent = WebResearchAgent()

# get user input
topic = input("Hello! What would you like me to research today?")

# run the agent
final_summary = agent.run(topic)
print(f"Final summary: {final_summary}")

Hello! What would you like me to research today?pizza

 Agent starting job for topic: pizza

[Thinking] I need to find information on the web for this topic. pizza I'll use the search_wikipedia tool.
Searching Wikipedia for: pizza
Unexpected Error: Expecting value: line 1 column 1 (char 0)
[Observation] The search tool failed. Aborting job.
Final summary: Unexpected Error: Expecting value: line 1 column 1 (char 0)
